In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

class MultilingualMusicRecommender:
    def __init__(self, df):
        self.df = df.copy() # Make a safe copy to avoid setting-with-copy warnings
        self.similarity_matrix = None
        self.combined_features = None
        
    def prepare_features(self):
        print("\nChecking for missing values (NaN) across features...")
        acoustic_cols = [
            'danceability', 'acousticness', 'energy', 'liveness', 
            'loudness', 'speechiness', 'tempo', 'Valence', 'key', 'mode'
        ]
        
        # 1. Self-Healing Imputation: Fix NaNs dynamically column by column
        for col in acoustic_cols:
            nan_count = self.df[col].isna().sum()
            if nan_count > 0:
                print(f"⚠️ Found {nan_count} missing values in '{col}'. Imputing with median value...")
                median_value = self.df[col].median()
                self.df[col] = self.df[col].fillna(median_value)
                
        # 2. Extract and scale acoustic features to uniform [0-1] range
        scaler = MinMaxScaler()
        scaled_acoustic = scaler.fit_transform(self.df[acoustic_cols])
        
        print("Vectorizing metadata text (Song Names + Singers)...")
        # Ensure text values contain no missing or non-string components
        self.df['song_name'] = self.df['song_name'].fillna("Unknown Song")
        self.df['singer'] = self.df['singer'].fillna("Unknown Singer")
        
        self.df['metadata_text'] = self.df['song_name'] + " " + self.df['singer'].str.replace('|', ' ', regex=False)
        
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        tfidf_matrix = tfidf.fit_transform(self.df['metadata_text'])
        
        # Give higher weighting to acoustic traits to favor cross-lingual audio vibe similarity
        acoustic_weight = 1.5
        weighted_acoustic = scaled_acoustic * acoustic_weight
        
        # Combine numerical arrays with textual matrices
        self.combined_features = hstack([weighted_acoustic, tfidf_matrix]).tocsr()
        print("Data preparation complete! Matrix format:", self.combined_features.shape)
        
    def compute_similarity(self):
        print("Computing similarity space matrix... (this might take a few seconds for 31,000+ tracks)")
        self.similarity_matrix = cosine_similarity(self.combined_features)
        print("🎉 Recommendation system is ready!")
        
    def get_recommendations(self, song_title, top_n=5, allow_cross_lingual=True, target_lang=None):
        # Match case-insensitive song titles
        idx_list = self.df[self.df['song_name'].str.lower() == song_title.lower()].index
        if len(idx_list) == 0:
            return f"Song '{song_title}' not found in the dataset."
        idx = idx_list[0]
        
        # Sort indices by cosine similarity scores
        sim_scores = list(enumerate(self.similarity_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        recommendations = []
        source_lang = self.df.iloc[idx]['language']
        
        for i, score in sim_scores:
            if i == idx:
                continue  # Skip the query song itself
                
            candidate_lang = self.df.iloc[i]['language']
            
            # Apply dynamic filtering flags
            if not allow_cross_lingual and candidate_lang != source_lang:
                continue
            if target_lang and candidate_lang.lower() != target_lang.lower():
                continue
                
            recommendations.append({
                'Song Name': self.df.iloc[i]['song_name'],
                'Singer(s)': self.df.iloc[i]['singer'],
                'Language': candidate_lang,
                'Vibe Match Score': round(score, 4)
            })
            
            if len(recommendations) == top_n:
                break
                
        return pd.DataFrame(recommendations)

# --- Initialize and Train ---
engine = MultilingualMusicRecommender(master_music_df)
engine.prepare_features()
engine.compute_similarity()


Checking for missing values (NaN) across features...
⚠️ Found 1 missing values in 'danceability'. Imputing with median value...
⚠️ Found 1 missing values in 'acousticness'. Imputing with median value...
⚠️ Found 1 missing values in 'energy'. Imputing with median value...
⚠️ Found 1 missing values in 'liveness'. Imputing with median value...
⚠️ Found 1 missing values in 'loudness'. Imputing with median value...
⚠️ Found 1 missing values in 'speechiness'. Imputing with median value...
⚠️ Found 1 missing values in 'tempo'. Imputing with median value...
⚠️ Found 1 missing values in 'Valence'. Imputing with median value...
⚠️ Found 1 missing values in 'key'. Imputing with median value...
⚠️ Found 1 missing values in 'mode'. Imputing with median value...
Vectorizing metadata text (Song Names + Singers)...
Data preparation complete! Matrix format: (31918, 5010)
Computing similarity space matrix... (this might take a few seconds for 31,000+ tracks)
🎉 Recommendation system is ready!
